# DataFlow360 — Fabric Lakehouse Medallion ETL

Attach this notebook to a Fabric Lakehouse before running. Upload the generated DataFlow360 CSV to `Files/landing/sales.csv`. The notebook preserves the raw input in Bronze, applies the repository's required-field, numeric, date, and `(order_id, line_number)` grain checks, upserts Silver and Gold Delta tables, and appends quality metrics. No Fabric execution is implied by running this notebook elsewhere.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

source_path = "Files/landing/sales.csv"
run_id = str(uuid4())
run_started_at = datetime.now(timezone.utc).isoformat()
print(f"DataFlow360 run_id={run_id}; source={source_path}")

## 1. Read and persist Bronze

Bronze is an append-only Delta table. The original source columns are retained alongside ingestion metadata for replay and lineage.

In [ ]:
raw = (
    spark.read.option("header", True).option("inferSchema", False).csv(source_path)
    .withColumn("_dataflow_run_id", F.lit(run_id))
    .withColumn("_ingested_at_utc", F.current_timestamp())
)
raw_count = raw.count()
raw.write.format("delta").mode("append").saveAsTable("bronze_sales")
print(f"Bronze rows written: {raw_count}")

## 2. Validate and classify source records

Repeated order IDs are not treated as duplicates when their line numbers differ. The source generator emits one line per order and therefore defaults missing `line_number` to 1.

In [ ]:
typed = (
    raw
    .withColumn("order_id", F.trim(F.col("order_id").cast("string")))
    .withColumn("line_number", F.coalesce(F.col("line_number").cast("int"), F.lit(1)))
    .withColumn("customer_id", F.trim(F.col("customer_id").cast("string")))
    .withColumn("customer_name", F.trim(F.col("customer_name").cast("string")))
    .withColumn("product_id", F.trim(F.col("product_id").cast("string")))
    .withColumn("product", F.trim(F.col("product").cast("string")))
    .withColumn("category", F.trim(F.col("category").cast("string")))
    .withColumn("supplier_id", F.trim(F.col("supplier_id").cast("string")))
    .withColumn("region", F.trim(F.col("region").cast("string")))
    .withColumn("_order_date_raw", F.trim(F.col("order_date").cast("string")))
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("decimal(12,2)"))
)
classified = typed.withColumn(
    "_quality_reason",
    F.when(F.col("order_id").isNull() | (F.col("order_id") == ""), "missing order_id")
     .when(F.col("product").isNull() | (F.col("product") == "") | F.col("category").isNull() | (F.col("category") == ""), "missing required product fields")
     .when(F.col("_order_date_raw").isNull() | (F.col("_order_date_raw") == ""), "missing order_date")
     .when(F.col("order_date").isNull(), "malformed order_date")
     .when(F.col("quantity").isNull(), "malformed quantity")
     .when(F.col("quantity") <= 0, "quantity must be positive")
     .when(F.col("unit_price").isNull(), "malformed unit_price")
     .when(F.col("unit_price") < 0, "unit_price must be non-negative")
     .when(F.col("line_number") < 1, "line_number must be positive")
)
valid_candidates = classified.filter(F.col("_quality_reason").isNull())
grain = Window.partitionBy("order_id", "line_number").orderBy(F.col("_ingested_at_utc").asc())
ranked = valid_candidates.withColumn("_grain_rank", F.row_number().over(grain))
duplicates = ranked.filter(F.col("_grain_rank") > 1).withColumn("_quality_reason", F.lit("duplicate order line"))
rejected = classified.filter(F.col("_quality_reason").isNotNull()).unionByName(duplicates, allowMissingColumns=True)
silver_rows = (
    ranked.filter(F.col("_grain_rank") == 1)
    .drop("_quality_reason", "_grain_rank", "_order_date_raw")
    .withColumn("total_amount", F.col("quantity") * F.col("unit_price"))
)
rejected_count = rejected.count()
accepted_count = silver_rows.count()
print(f"Silver candidates accepted={accepted_count}; quarantined/rejected={rejected_count}")

## 3. Persist quality outcomes and incrementally MERGE Silver

Silver's key is `(order_id, line_number)`. Reprocessing the same source is idempotent; changed values update the existing Delta row.

In [ ]:
reason_counts = {row["_quality_reason"]: row["count"] for row in rejected.groupBy("_quality_reason").count().collect()}
missing_count = sum(count for reason, count in reason_counts.items() if reason.startswith("missing"))
malformed_count = sum(count for reason, count in reason_counts.items() if reason.startswith("malformed"))
invalid_count = sum(count for reason, count in reason_counts.items() if reason.startswith("quantity must") or reason.startswith("unit_price must") or reason.startswith("line_number"))
duplicate_count = reason_counts.get("duplicate order line", 0)
quality_metrics = spark.createDataFrame(
    [(run_id, raw_count, accepted_count, rejected_count, missing_count, malformed_count, invalid_count, duplicate_count, run_started_at)],
    ["run_id", "raw_rows", "accepted_rows", "rejected_rows", "missing_rows", "malformed_rows", "invalid_rows", "duplicate_rows", "run_started_at_utc"],
)
quality_metrics.write.format("delta").mode("append").saveAsTable("dataflow_quality_runs")
rejected.withColumn("_quality_run_id", F.lit(run_id)).write.format("delta").mode("append").saveAsTable("dataflow_quarantine")

if not spark.catalog.tableExists("silver_sales"):
    silver_rows.write.format("delta").mode("overwrite").saveAsTable("silver_sales")
else:
    DeltaTable.forName(spark, "silver_sales").alias("target").merge(
        silver_rows.alias("source"),
        "target.order_id = source.order_id AND target.line_number = source.line_number",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f"Silver upsert complete; current rows={spark.table('silver_sales').count()}")

## 4. Build and MERGE Gold analytics

Gold groups current Silver facts by date, region, and category. The source is recomputed from Silver so updated records move between groups correctly; stale aggregate groups are removed by the Delta MERGE.

In [ ]:
gold_rows = (
    spark.table("silver_sales")
    .groupBy("order_date", "region", "category")
    .agg(
        F.count(F.lit(1)).alias("sales_lines"),
        F.sum("quantity").alias("units_sold"),
        F.sum("total_amount").alias("revenue"),
    )
    .withColumn("_refreshed_at_utc", F.current_timestamp())
)
if not spark.catalog.tableExists("gold_sales_daily"):
    gold_rows.write.format("delta").mode("overwrite").saveAsTable("gold_sales_daily")
else:
    DeltaTable.forName(spark, "gold_sales_daily").alias("target").merge(
        gold_rows.alias("source"),
        "target.order_date = source.order_date AND target.region = source.region AND target.category = source.category",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().whenNotMatchedBySourceDelete().execute()

display(gold_rows.orderBy(F.col("order_date").desc()).limit(25))

## 5. Business-level result and quality summary

In [ ]:
display(
    spark.table("gold_sales_daily")
    .groupBy("category")
    .agg(F.sum("sales_lines").alias("sales_lines"), F.sum("units_sold").alias("units_sold"), F.sum("revenue").alias("revenue"))
    .orderBy(F.col("revenue").desc())
)
display(spark.table("dataflow_quality_runs").filter(F.col("run_id") == run_id))